In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [4]:
Logger.info("="*100)
Logger.Slack.info('Starting Sur Source Summary Ingester')                   

In [5]:
#Get customer list
customer_list = Query(query = "SELECT * FROM KPI_Customer WHERE DBLocation IS NOT 'Unknown' AND Name = 'Cadent'").execute(KPIHub_Conn)

#Set the time window for the update
current_date = date.today()
update_window = current_date - timedelta(days=UPDATE_WINDOW_DAYS)

In [6]:
# Define a function to determine if survey is in 'day' or 'night'
def get_day_night(start_time, sunrise, sunset):
    # start_time should be a datetime.time object
    hour = start_time.hour
    if sunrise <= hour < sunset:
        return 'Day'
    else:
        return 'Night'

def set_actie_idle(speed, speed_threshold):
    if speed < speed_threshold:
        return 'Idle'
    else:
        return 'Active'

def segment_summary_apply(df):
    return pd.Series({
        'DaySegments': (df['DayNight'] == 'Day').sum(),
        'NightSegments': (df['DayNight'] == 'Night').sum(),
        'ActiveSegments': (df['ActiveIdle'] == 'Active').sum(),
        'IdleSegments': (df['ActiveIdle'] == 'Idle').sum(),
        'TotalSegments': len(df),
        'TotalKilometers': df['LengthMeters'].sum() / 1000,
        'DayKilometers': df.loc[df['DayNight'] == 'Day', 'LengthMeters'].sum() / 1000,
        'NightKilometers': df.loc[df['DayNight'] == 'Night', 'LengthMeters'].sum() / 1000,
        'SegmentDurationMinutes': df['DurationSeconds'].sum() / 60,
        'IdleTimeMinutes': df.loc[df['ActiveIdle'] == 'Idle', 'DurationSeconds'].sum() / 60,
        'ActiveTimeMinutes': df.loc[df['ActiveIdle'] == 'Active', 'DurationSeconds'].sum() / 60,
        'AvgSpeedKm': 3.6*df['CarSpeedMedian'].mean()

    })

def survey_summary_apply(row):
    return pd.Series({
        'SurveyId': row['SurveyId'],
        'SurveyorUnit': row['SurveyorUnit'],
        'SurveyDurationMinutes': row['DurationMinutes'],
        'ReportId': row['ReportId'],
        'StartHour': row['StartHour'],
        'StartTime': row['StartTime'],
        'StartEpoch': row['StartEpoch'],
        'EndTime': row['EndTime'],
        'EndEpoch': row['EndEpoch'],
        'StartDay': row['StartDay'],
        'EndDay': row['EndDay'],
        'LateralRotation': row['LateralRotation'],
        'NumberOfPeaks': row['NumberOfPeaks']
    })

In [7]:
for _, row in customer_list.iterrows():
    customer_name = row['Name']
    customer_id = row['CustomerId']
    customer_db = row['DBLocation']

    Logger.info(f"Processing customer: {customer_name}")
    Logger.info(f"Getting reports from {update_window} to {current_date}")
    #Query the last report
    reports = Query(
        f"""
        SELECT ReportId, ReportDate, LastUpdated FROM KPI_ReportSummary
        WHERE CustomerId = '{customer_id}'
        ORDER BY LastUpdated DESC
        """
    ).execute(KPIHub_Conn)

    survey_count = Query(
        f"""
        SELECT COUNT(*) as SurveyCount
        FROM KPI_SurveySummary
        WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}')
        """
    ).execute(KPIHub_Conn)

    if len(reports) > 0:
        if (survey_count.iloc[0]['SurveyCount']) > 0:
            process_surveys = True
            starting_date = pd.to_datetime(update_window)
            Logger.info(f"Getting emissions from {update_window} to {current_date}")

        else:
            Logger.info(f"No emissions found, processing")
            starting_date = datetime(STARTING_YEAR, 1, 1)
            process_surveys = True
    else:
        Logger.info(f"No reports found, skipping")
        process_surveys = False

    if process_surveys:
        # Ensure the ReportDate values are in datetime format before comparison,
        # handling both with and without microseconds (mixed formats)
        reports['ReportDate'] = pd.to_datetime(reports['ReportDate'], format='mixed')

        reports_to_query = reports[reports['ReportDate'] >= starting_date]
        reports_to_query.db.set_query(query_surveys_table(report_table="#TempReports"))
        surveys = reports_to_query.db.execute(CONN_DICT[customer_db], source_col = 'ReportId', temp_table_name = '#TempReports')
        surveys.db.set_query(query_segments_table(survey_table="#TempSurvey"))
        Logger.info(f"Surveys from LSDB: {len(surveys)}")
        segments = surveys.db.execute(CONN_DICT[customer_db], source_col = 'SurveyId', temp_table_name = '#TempSurvey')
        Logger.info(f"Segments from LSDB: {len(segments)}")
        # Convert StartEpoch to datetime (time only, no date)
        segments['StartTime'] = pd.to_datetime(segments['StartEpoch'], unit='s').dt.time
        segments['StartDate'] = pd.to_datetime(segments['StartEpoch'], unit='s')
        segments['DayNight'] = segments['StartTime'].apply(lambda t: get_day_night(t, SUNRISE_TIME, SUNSET_TIME))
        segments['ActiveIdle'] = segments['CarSpeedMedian'].apply(lambda x: set_actie_idle(x, SPEED_THRESHOLD))

        # Set the starting time as a datetime object
        surveys['StartHour'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.hour
        surveys['StartTime'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.time
        surveys['EndHour'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.hour
        surveys['EndTime'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.time
        surveys['StartDay'] = pd.to_datetime(surveys['StartEpoch'], unit='s').dt.date
        surveys['EndDay'] = pd.to_datetime(surveys['EndEpoch'], unit='s').dt.date

        # Calculate the duration (in minutes) between StartEpoch and EndEpoch for each survey
        surveys['DurationMinutes'] = (
            surveys['EndEpoch'] - surveys['StartEpoch']
        ) / 60

        # Apply the function row-wise (axis=1) to build a DataFrame summary
        survey_summary = surveys.apply(survey_summary_apply, axis=1)
        segment_summary = segments.groupby("SurveyId").apply(segment_summary_apply)
        survey_summary = pd.merge(survey_summary,segment_summary,on="SurveyId",how="left")

        upload = survey_summary.reset_index(drop=True)
        upload.fillna(0, inplace=True)
        upload['LastUpdated'] = datetime.now()

        Logger.info(f"Reports from LSDB: {len(upload)}")
        KPI_SurveySummary.update_table(arguments = {'db_path': DB_PATH, 'DataFrame': upload, 'PrimaryKey': ['SurveyId','ReportId']})
        df_kpi = KPI_SurveySummary.query_table(arguments = {'db_path': DB_PATH})
        Logger.info(f"Reports from KPI_SurveySummary: {len(df_kpi)}")

    else:
        Logger.info(f"No reports found, skipping")

       
